# Proof Pilot — DFlash deploy smoke test (GQA-packed triton attn + DFlash spec-v2 + gptq)

End-to-end check on a single RTX PRO 6000 (sm120), **offline**: boots the **w4a8 + DFlash**
32B sglang server from the `proof-pilot-env` dataset + the `opd-32b-...-gptq-w4a16` target model
+ the `dflash-32b-draft-v2test` **draft** model. DFlash spec-decoding uses the GQA-packed
split-K extend/verify kernel (`NUM_DRAFT=8` -> M=64 fast path). Then a quick chat generation and
ONE problem through the full prove/verify/refine/select loop.

**Add all three:** the `proof-pilot-env` dataset, the gptq model, **and** the dflash draft model.
Then *Run All*. Green ALL PASS = the DFlash deploy path works.

In [ ]:
# --- 1. locate + extract the env archive into writable scratch (Kaggle input is read-only) ---
import os, subprocess, glob, re, json, time, textwrap, shutil, urllib.request

WORK = os.environ.get("WORK", "/tmp/pp")
VENV, PYBASE = f"{WORK}/venv", f"{WORK}/pybase"
# persistent, DOWNLOADABLE log/output dir on Kaggle (/tmp is wiped at session end); falls back to
# WORK locally. server.log / humming stderr / proof traces land here so a crash stays debuggable.
LOGDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else WORK
os.makedirs(WORK, exist_ok=True); os.makedirs(LOGDIR, exist_ok=True)
print("LOGDIR (persistent) =", LOGDIR)

def find_archive():
    cands = ([os.environ["DS_ENV"]] if os.environ.get("DS_ENV") else []) + [
        "/kaggle/input/proof-pilot-env",
        "/kaggle/input/datasets/threerabbits/proof-pilot-env"]
    for d in cands:
        for p in ("proof-pilot-env.bin", "proof-pilot-env.tar.*", "*.bin", "*.tar.gz", "*.tar.zst"):
            g = sorted(glob.glob(f"{d}/{p}"))
            if g: return d, g[0]
    g = [x for x in sorted(glob.glob("/kaggle/input/**/proof-pilot-env.*", recursive=True))
         if x.endswith((".bin", ".tar.gz", ".tar.zst"))]
    return (os.path.dirname(g[0]), g[0]) if g else (None, None)

if not os.path.exists(f"{VENV}/bin/python"):
    DS_ENV, arc = find_archive()
    assert arc, "env archive not found — add the proof-pilot-env dataset"
    magic = open(arc, "rb").read(4)
    print("env archive =", arc, "| magic =", magic.hex())
    if magic[:2] == b"\x1f\x8b":
        subprocess.run(["tar", "-xzf", arc, "-C", WORK, "--strip-components=1"], check=True)
    elif magic == b"\x28\xb5\x2f\xfd":
        assert shutil.which("zstd"), "zstd archive but no zstd binary"
        subprocess.run(["tar", "-x", "-I", "zstd -d --long=31", "-f", arc, "-C", WORK, "--strip-components=1"], check=True)
    else:
        raise SystemExit(f"unknown archive magic {magic!r}")
assert os.path.exists(f"{VENV}/bin/python"), "venv missing after extract"
print("env staged ->", WORK)

In [ ]:
# --- 2. relocate the venv (stdlib in pybase) + restore warm caches + export paths ---
cfg = f"{VENV}/pyvenv.cfg"
txt = re.sub(r"(?m)^home = .*$", f"home = {PYBASE}/bin", open(cfg).read())
open(cfg, "w").write(txt)
HOME = os.path.expanduser("~")
for src, dst in [(f"{WORK}/flashinfer_cache", f"{HOME}/.cache/flashinfer"),
                 (f"{WORK}/humming_cache",    f"{HOME}/.humming/cache")]:
    if os.path.isdir(src):
        os.makedirs(dst, exist_ok=True); subprocess.run(["cp", "-rn", f"{src}/.", f"{dst}/"])
REPO = f"{WORK}/proof-pilot"
print("pyvenv:", next(l for l in txt.splitlines() if l.startswith("home")), "| REPO:", REPO)

In [ ]:
# --- 3. locate the gptq TARGET (+ writable SWA view) and the DFlash DRAFT model ---
# target = the model with a quantization_config (compressed-tensors gptq); draft = the other one
# (qwen3, no quant). The target gets a symlink view with a writable config so enable_swa_config can
# patch it (Kaggle model mount is read-only). The draft is loaded read-only as-is (no config edit).
def find_models():
    md, dd = os.environ.get("MODEL_DIR"), os.environ.get("DRAFT_DIR")
    if md and dd: return md, dd
    dirs = sorted(set(os.path.dirname(c) for c in glob.glob("/kaggle/input/**/config.json", recursive=True)
                  if glob.glob(os.path.join(os.path.dirname(c), "*.safetensors*"))))
    for d in dirs:
        try: cfg = json.load(open(os.path.join(d, "config.json")))
        except Exception: continue
        if cfg.get("quantization_config"): md = md or d           # gptq target
        else: dd = dd or d                                        # unquantized draft
    return md, dd

MODEL_SRC, DRAFT = find_models()
assert MODEL_SRC, "gptq target not found — add the opd-32b-...-gptq-w4a16 model"
assert DRAFT, "dflash draft not found — add the dflash-32b-draft-v2test model"
MODEL = f"{WORK}/model_view"; os.makedirs(MODEL, exist_ok=True)
for f in os.listdir(MODEL_SRC):
    dst = f"{MODEL}/{f}"
    if not os.path.lexists(dst): os.symlink(f"{MODEL_SRC}/{f}", dst)
cfgp = f"{MODEL}/config.json"
if os.path.islink(cfgp): os.remove(cfgp)
shutil.copy(f"{MODEL_SRC}/config.json", cfgp)                       # writable copy
subprocess.run([f"{VENV}/bin/python", f"{REPO}/kaggle/serve/enable_swa_config.py", MODEL], check=True)
print("target src:", MODEL_SRC)
print("target view:", MODEL)
print("draft     :", DRAFT)

In [ ]:
# --- 3b. patch the venv weight loader to SKIP baked fp8-KV scales (k_scale/v_scale) ---
# (also baked into the shipped venv; this is a no-op belt-and-suspenders for older bundles)
import py_compile
olmo2 = glob.glob(f"{VENV}/lib/python*/site-packages/sglang/srt/models/olmo2.py")[0]
src = open(olmo2).read()
if "k_scale" in src:
    print("olmo2.py already skips k_scale/v_scale (no-op)")
else:
    anchor = "        for name, loaded_weight in weights:\n"
    guard = ('            if name.endswith((".k_scale", ".v_scale")) and name not in params_dict:\n'
             "                continue\n")
    assert src.count(anchor) == 1, f"olmo2.py load-loop anchor not unique ({src.count(anchor)})"
    open(olmo2, "w").write(src.replace(anchor, anchor + guard, 1))
    print("patched olmo2.py: skip baked k_scale/v_scale")
py_compile.compile(olmo2, doraise=True)
print("olmo2.py OK")

In [ ]:
# --- 3c. expose CCCL (cuda/std/*) headers to humming's NVRTC (serve_final also symlinks these;
#         kept here for older bundles). ---
_cccl = textwrap.dedent(f"""
    import os, sys, glob
    os.environ.setdefault("HUMMING_PATH", "{WORK}")
    sys.path.insert(0, os.environ["HUMMING_PATH"])
    from humming.utils.cuda import filter_cuda_paths
    fi = sorted(glob.glob(os.path.join(sys.prefix, "lib/python*/site-packages/flashinfer/data/cccl/libcudacxx/include")))
    assert fi, "flashinfer libcudacxx (CCCL) headers not found in venv"
    fi = fi[0]
    roots = filter_cuda_paths(required_headers=["cuda_runtime.h"])["include_paths"]
    made = []
    for p in roots:
        if not os.path.isdir(p):
            continue
        cccl = os.path.join(p, "cccl")
        if not os.path.exists(os.path.join(cccl, "cuda", "std", "cstdint")) and not os.path.exists(cccl):
            try:
                os.symlink(fi, cccl); made.append(cccl)
            except OSError as e:
                print("skip", cccl, e)
    reach = any(os.path.exists(os.path.join(p, "cccl", "cuda/std/cstdint"))
                or os.path.exists(os.path.join(p, "cuda/std/cstdint")) for p in roots)
    print("cuda include roots:", roots, "| cccl reachable:", reach)
    assert reach, "CCCL still not reachable -- humming NVRTC will fail"
""")
r = subprocess.run([f"{VENV}/bin/python", "-c", _cccl],
                   env=dict(os.environ, PYBASE=PYBASE), capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print("--- stderr ---\n", r.stderr[-2500:]); raise SystemExit("CCCL setup failed")

In [ ]:
# --- 4. launch the w4a8 + DFlash 32B sglang server (offline) and wait for health ---
CONFIG, PORT = "w4a8-dflash", 30000
nvrtc_lib = f"{VENV}/lib/python3.12/site-packages/nvidia/cu13/lib"   # libnvrtc-builtins.so.13.0
link_dir = f"{WORK}/_link"; os.makedirs(link_dir, exist_ok=True)     # libcuda.so for flashinfer JIT link
_libcuda = next((p for p in (
    glob.glob("/usr/local/cuda*/targets/*/lib/stubs/libcuda.so")
    + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so")
    + glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so.1")
    + glob.glob("/usr/local/cuda*/compat/libcuda.so*")) if os.path.exists(p)), None)
if _libcuda and not os.path.lexists(f"{link_dir}/libcuda.so"):
    os.symlink(_libcuda, f"{link_dir}/libcuda.so")
print("libcuda for flashinfer JIT link:", _libcuda)
env = dict(os.environ, CONFIG=CONFIG, PORT=str(PORT), CUDA_VISIBLE_DEVICES="0",
           LIBRARY_PATH=link_dir + ((":" + os.environ["LIBRARY_PATH"]) if os.environ.get("LIBRARY_PATH") else ""),
           VENV=VENV, REPO=REPO, GPTQ=MODEL, DRAFT=DRAFT, HUMMING_DIR=WORK,
           HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
           FLASHINFER_CUDA_ARCH_LIST="12.0f",   # "12.0" raises on Kaggle's <12.9 driver; "f" skips that
           LD_LIBRARY_PATH=nvrtc_lib + ((":" + os.environ["LD_LIBRARY_PATH"]) if os.environ.get("LD_LIBRARY_PATH") else ""))

SERVER_LOG = f"{LOGDIR}/server.log"
def _dump_diag():
    print("=== server.log:", SERVER_LOG, "(tail) ===\n", open(SERVER_LOG).read()[-8000:])
    hd = f"{LOGDIR}/humming_stderr"; os.makedirs(hd, exist_ok=True)
    for f in glob.glob(os.path.expanduser("~/.humming/cache/*/stderr.log")):
        t = open(f).read().strip()
        if t:
            shutil.copy(f, f"{hd}/{os.path.basename(os.path.dirname(f))}.log")
            print(f"=== {f} ===\n", t[-2500:])

logf = open(SERVER_LOG, "w")
print("sglang server log ->", SERVER_LOG)
srv = subprocess.Popen(["bash", f"{REPO}/kaggle/serve/serve_final.sh"],
                       env=env, stdout=logf, stderr=subprocess.STDOUT)
print("server starting (32B + draft load + humming JIT + cuda-graph; can take several minutes)...")
ready = False
for _ in range(216):                                   # up to ~18 min (draft adds load/capture)
    try:
        if urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5).status == 200:
            ready = True; break
    except Exception: pass
    if srv.poll() is not None:
        print("SERVER DIED:"); _dump_diag(); raise SystemExit(1)
    time.sleep(5)
if not ready:
    _dump_diag(); raise SystemExit("server did not become healthy in time")
print("server READY on :%d (w4a8 + DFlash)" % PORT)

In [ ]:
# --- 5. smoke generation: does the served model produce coherent math text? ---
base = f"http://127.0.0.1:{PORT}"
model_id = json.load(urllib.request.urlopen(f"{base}/v1/models", timeout=10))["data"][0]["id"]
payload = json.dumps({
    "model": model_id,
    "messages": [{"role": "user", "content": "Prove that there are infinitely many prime numbers. Keep it short."}],
    "temperature": 0.7, "max_tokens": 600,
}).encode()
req = urllib.request.Request(f"{base}/v1/chat/completions", data=payload,
                             headers={"Content-Type": "application/json"})
r = json.load(urllib.request.urlopen(req, timeout=600))
msg = r["choices"][0]["message"]
text = (msg.get("content") or "") + (msg.get("reasoning_content") or "")
print("served model:", model_id)
print("--- generation (first 800 chars) ---\n", text[:800])
smoke_ok = len(text.strip()) > 80 and ("prime" in text.lower())
print("\nsmoke_ok:", smoke_ok)

In [ ]:
# --- 6. one problem through the full prove/verify/refine/select loop (short budget for the test) ---
BUDGET_S = int(os.environ.get("BUDGET_S", "420"))      # raise to 3300 for the real submission
csv_in, csv_out, logd = f"{WORK}/sample.csv", f"{LOGDIR}/submission.csv", f"{LOGDIR}/logs"
with open(csv_in, "w") as f:
    f.write("id,problem\n")
    f.write('1,"Prove that for every positive integer n, the sum 1+2+...+n equals n(n+1)/2."\n')
cmd = [f"{VENV}/bin/python", f"{REPO}/kaggle/notebook/run.py",
       "--model_path", MODEL, "--input_csv", csv_in, "--output_csv", csv_out, "--logdir", logd,
       "--base-url", base, "--budget-s", str(BUDGET_S), "--select-reserve", "120",
       "--init-provers", "4", "--verify-k", "2", "--refine-inputs", "3", "--selectors", "3",
       "--max-tokens", "32000", "--concurrency", "8", "--call-cap", "16000"]
print("running proof loop, budget", BUDGET_S, "s ...")
subprocess.run(cmd, env=env, check=True)
import csv as _csv
rows = list(_csv.DictReader(open(csv_out)))
proof = rows[0]["prediction"] if rows else ""
print("--- proof (first 600 chars) ---\n", proof[:600])
loop_ok = len(proof.strip()) > 120
print("\nloop_ok:", loop_ok, "| proof_len:", len(proof))

In [ ]:
# --- 7. summary + shut the server down ---
print("=== DFLASH DEPLOY SMOKE TEST ===")
for k, v in [("server_ready", ready), ("smoke_gen", smoke_ok), ("proof_loop", loop_ok)]:
    print(("  PASS" if v else "  FAIL"), k)
srv.terminate()
try: srv.wait(timeout=30)
except Exception: srv.kill()
ok = ready and smoke_ok and loop_ok
print("\n" + ("ALL PASS - w4a8 + DFlash deploy path works end-to-end" if ok else "FAILED - see logs above"))
assert ok, "dflash deploy smoke test failed"